### LLM Chain 만들기


## 1. 환경 구성

### 1) 라이브러리 설치

In [ ]:
# uv add python-dotenv langchain langchain-openai

Note: you may need to restart the kernel to use updated packages.


c:\rookies6\langchain\mylangchin-uv-project\.venv\Scripts\python.exe: No module named uv


### 2) GROQ 인증키 설정

In [2]:
from dotenv import load_dotenv
import os

# .env 파일을 불러와서 환경 변수로 설정
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
print(GROQ_API_KEY[:3])

gsk


In [3]:
import langchain
print(langchain.__version__)

1.4.3


In [ ]:
from langchain_openai import ChatOpenAI

# Groq(OpenAI 호환) 공통 설정 — 모델/온도 변경은 이곳만 수정
GROQ_BASE_URL = "https://api.groq.com/openai/v1"  # Groq API 엔드포인트
GROQ_MODEL = "openai/gpt-oss-120b"  # 대안: "qwen/qwen3.8-27b"
#MOVIE_MODEL = "qwen/qwen3.8-27b"  # 영화 추천 예제용 
TEMPERATURE = 0.7


def get_llm(model: str = GROQ_MODEL, temperature: float = TEMPERATURE) -> ChatOpenAI:
    """Groq 엔드포인트를 사용하는 ChatOpenAI 인스턴스를 생성합니다."""
    return ChatOpenAI(
        api_key=GROQ_API_KEY,
        base_url=GROQ_BASE_URL,
        model=model,
        temperature=temperature,
    )


# 기본 llm (이후 셀에서 재사용)
llm = get_llm()
print(f'model:name')

## 2. LLM Chain

### 1) Prompt + LLM


In [5]:
# model: 공통 설정 셀의 llm 사용

# chain 실행
result = llm.invoke("인공지능 모델의 학습 원리에 대하여 쉽게 설명해 주세요.")
print(type(result))
print(result)

<class 'langchain_core.messages.ai.AIMessage'>
content="## 인공지능 모델, 특히 **머신러닝·딥러닝** 모델이 어떻게 “배우는지”를 쉽게 풀어볼게요.\n\n---\n\n### 1️⃣ 모델은 **수학 공식**(함수)이라고 생각하면 돼요  \n- 입력(예: 사진, 텍스트, 숫자) → **예측값**(예: 고양이·강아지 구분, 다음 단어)  \n- 이때 **가중치(Weight)** 라는 숫자들이 함수 안에 들어가 있어요. 가중치가 바로 모델이 “배우는” 부분이에요.\n\n---\n\n### 2️⃣ **데이터**가 학습의 원동력  \n| 종류 | 역할 |\n|------|------|\n| **입력 데이터** (X) | 모델에 보여줄 예시(예: 사진) |\n| **정답 레이블** (Y) | “정답”을 알려주는 라벨(예: 고양이=0, 강아지=1) |\n\n> **예시**: 사진 1,000장을 보여주고, 각각 “고양이” 혹은 “강아지” 라는 라벨을 붙여 놓은 것이 학습 데이터예요.\n\n---\n\n### 3️⃣ **손실 함수(Loss, Cost)** → “얼마나 틀렸나?”를 숫자로 나타냄  \n- 모델이 내놓은 예측값과 실제 정답을 비교해서 **오차**를 계산합니다.  \n- 흔히 쓰는 손실 함수:  \n  - **MSE(Mean Squared Error)** – 회귀(숫자 예측) 문제  \n  - **Cross‑Entropy** – 분류(고양이·강아지) 문제  \n\n> 손실이 **클수록** 모델이 많이 틀렸다는 뜻이에요. 학습 목표는 이 손실을 **작게** 만드는 것!\n\n---\n\n### 4️⃣ **최적화(Optimization) → 가중치를 어떻게 바꿀까?**  \n1. **경사 하강법(Gradient Descent)**  \n   - 손실 함수의 **기울기(gradient)** 를 구합니다.  \n   - 기울기가 가리키는 방향은 “손실이 가장 크게 증가하는 방향”.  \n   - 그래서 **반대 방향**(기울기

In [6]:
print(result.content)

## 인공지능 모델, 특히 **머신러닝·딥러닝** 모델이 어떻게 “배우는지”를 쉽게 풀어볼게요.

---

### 1️⃣ 모델은 **수학 공식**(함수)이라고 생각하면 돼요  
- 입력(예: 사진, 텍스트, 숫자) → **예측값**(예: 고양이·강아지 구분, 다음 단어)  
- 이때 **가중치(Weight)** 라는 숫자들이 함수 안에 들어가 있어요. 가중치가 바로 모델이 “배우는” 부분이에요.

---

### 2️⃣ **데이터**가 학습의 원동력  
| 종류 | 역할 |
|------|------|
| **입력 데이터** (X) | 모델에 보여줄 예시(예: 사진) |
| **정답 레이블** (Y) | “정답”을 알려주는 라벨(예: 고양이=0, 강아지=1) |

> **예시**: 사진 1,000장을 보여주고, 각각 “고양이” 혹은 “강아지” 라는 라벨을 붙여 놓은 것이 학습 데이터예요.

---

### 3️⃣ **손실 함수(Loss, Cost)** → “얼마나 틀렸나?”를 숫자로 나타냄  
- 모델이 내놓은 예측값과 실제 정답을 비교해서 **오차**를 계산합니다.  
- 흔히 쓰는 손실 함수:  
  - **MSE(Mean Squared Error)** – 회귀(숫자 예측) 문제  
  - **Cross‑Entropy** – 분류(고양이·강아지) 문제  

> 손실이 **클수록** 모델이 많이 틀렸다는 뜻이에요. 학습 목표는 이 손실을 **작게** 만드는 것!

---

### 4️⃣ **최적화(Optimization) → 가중치를 어떻게 바꿀까?**  
1. **경사 하강법(Gradient Descent)**  
   - 손실 함수의 **기울기(gradient)** 를 구합니다.  
   - 기울기가 가리키는 방향은 “손실이 가장 크게 증가하는 방향”.  
   - 그래서 **반대 방향**(기울기 × 학습률) 으로 조금씩 가중치를 이동시켜 손실을 줄여요.

2. **학습률(Learning Rate)**  
   - 한 번에 움직이는 거리. 너무 크면 지나

### 2) PromptTemplate + LLM

In [7]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template("You are an expert in AI Expert. Answer the question.\
                                       <Question>: {input}에 대해 쉽게 설명해주세요.")

print(type(prompt))
print(prompt)

<class 'langchain_core.prompts.prompt.PromptTemplate'>
input_variables=['input'] input_types={} partial_variables={} template='You are an expert in AI Expert. Answer the question.                                       <Question>: {input}에 대해 쉽게 설명해주세요.'


In [8]:
# llm: 공통 설정 셀의 llm 사용
# chain 연결 (LCEL)
chain = prompt | llm
print(type(chain))

# chain 호출
result = chain.invoke({"input": "인공지능 모델의 학습 원리"})
print(type(result))
print(result)

<class 'langchain_core.runnables.base.RunnableSequence'>
<class 'langchain_core.messages.ai.AIMessage'>
content='## 인공지능 모델은 어떻게 배우나요?  \n(쉽게 이해할 수 있도록 비유와 핵심 개념을 함께 설명합니다)\n\n---\n\n### 1. **데이터 → 경험 → 지식**\n| 단계 | 비유 | 설명 |\n|------|------|------|\n| **입력 데이터** | **학생이 교과서** | 모델에게 “문제와 정답”이 들어 있는 많은 예시(이미지, 텍스트, 소리 등)를 보여줍니다. |\n| **학습** | **숙제와 시험** | 모델은 주어진 예시를 가지고 ‘패턴’을 찾아내려고 연습합니다. |\n| **예측** | **시험문제 풀기** | 학습이 끝난 뒤 새로운 데이터를 보았을 때 정답을 맞추는 능력이 생깁니다. |\n\n---\n\n### 2. **핵심 요소 4가지**\n\n| 요소 | 역할 | 쉬운 비유 |\n|------|------|----------|\n| **손실 함수(Loss)** | “얼마나 틀렸는가”를 수치화 | 시험 점수: 오답이 많을수록 점수가 낮음 |\n| **모델 파라미터(가중치, 편향)** | 문제를 푸는 ‘공식’ | 학생이 외운 공식들 |\n| **옵티마이저(Optimizer)** | 파라미터를 조금씩 고치는 방법 | 공부하면서 틀린 문제를 다시 풀어보는 전략 |\n| **역전파(Back‑propagation)** | 손실을 줄이기 위해 파라미터를 어떻게 바꿀지 알려줌 | “왜 틀렸는지”를 거꾸로 추적해 정답을 찾는 과정 |\n\n---\n\n### 3. **학습 흐름을 단계별로 풀어보면**\n\n1️⃣ **예시 제시**  \n   - 입력(예: 고양이 사진) + 정답(‘고양이’)을 모델에 줍니다.\n\n2️⃣ **예측**  \n   - 현재 파라미터(공식)로 “이 사진이 뭐라고 생각하나요?” 라고 물어봅니다.  \n   - 모델

In [9]:
print(result.content)

## 인공지능 모델은 어떻게 배우나요?  
(쉽게 이해할 수 있도록 비유와 핵심 개념을 함께 설명합니다)

---

### 1. **데이터 → 경험 → 지식**
| 단계 | 비유 | 설명 |
|------|------|------|
| **입력 데이터** | **학생이 교과서** | 모델에게 “문제와 정답”이 들어 있는 많은 예시(이미지, 텍스트, 소리 등)를 보여줍니다. |
| **학습** | **숙제와 시험** | 모델은 주어진 예시를 가지고 ‘패턴’을 찾아내려고 연습합니다. |
| **예측** | **시험문제 풀기** | 학습이 끝난 뒤 새로운 데이터를 보았을 때 정답을 맞추는 능력이 생깁니다. |

---

### 2. **핵심 요소 4가지**

| 요소 | 역할 | 쉬운 비유 |
|------|------|----------|
| **손실 함수(Loss)** | “얼마나 틀렸는가”를 수치화 | 시험 점수: 오답이 많을수록 점수가 낮음 |
| **모델 파라미터(가중치, 편향)** | 문제를 푸는 ‘공식’ | 학생이 외운 공식들 |
| **옵티마이저(Optimizer)** | 파라미터를 조금씩 고치는 방법 | 공부하면서 틀린 문제를 다시 풀어보는 전략 |
| **역전파(Back‑propagation)** | 손실을 줄이기 위해 파라미터를 어떻게 바꿀지 알려줌 | “왜 틀렸는지”를 거꾸로 추적해 정답을 찾는 과정 |

---

### 3. **학습 흐름을 단계별로 풀어보면**

1️⃣ **예시 제시**  
   - 입력(예: 고양이 사진) + 정답(‘고양이’)을 모델에 줍니다.

2️⃣ **예측**  
   - 현재 파라미터(공식)로 “이 사진이 뭐라고 생각하나요?” 라고 물어봅니다.  
   - 모델은 확률값(예: 고양이 0.6, 강아지 0.3, 자동차 0.1)으로 대답합니다.

3️⃣ **손실 계산**  
   - 정답(‘고양이’)과 모델이 낸 확률 사이의 차이를 계산합니다.  
   - 차이가 크면 손실값이 크게 나오고, 작으면 작게 나옵니다.

4️

### 3) PromptTemplate + LLM(invoke()) + StrOutputParser

In [10]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 컴포넌트 정의 (llm 은 공통 설정 셀의 llm 사용)
prompt = PromptTemplate.from_template("You are an expert in AI Expert. \
                                      Answer the question. <Question>: {input}에 대해 쉽게 설명해주세요.")

output_parser = StrOutputParser()

# 2. chain 생성 (LCEL)
chain = prompt | llm | output_parser
print(type(chain))

# 3. chain의 invoke 호출
result = chain.invoke({"input": "인공지능 모델의 학습 원리"})
print(type(result))
print(result)

<class 'langchain_core.runnables.base.RunnableSequence'>
<class 'langchain_core.messages.base.TextAccessor'>
### 인공지능 모델(특히 딥러닝)의 학습 원리를 쉽게 풀어 설명하면

| 단계 | 무슨 일? | 비유·예시 |
|------|----------|----------|
| **1. 데이터 모으기** | 모델이 배울 **입력‑정답 쌍**(예: 사진과 그 사진에 대한 라벨)을 준비합니다. | **학생이 교과서와 정답지를 받는 것** |
| **2. 모델 만들기** | 입력을 받아서 출력을 내는 **수식(네트워크 구조)**을 정의합니다. 보통 여러 층(layer)으로 이루어진 ‘인공 뉴런’들의 연결망. | **학생이 문제를 푸는 방법(공식, 절차)을 정하는 것** |
| **3. 추론(예측)** | 현재 모델(초기에는 무작위 가중치)로 입력을 넣어 출력을 계산합니다. | **학생이 아직 공부를 안 해서 엉뚱한 답을 내는 상황** |
| **4. 오차(손실) 계산** | 모델이 낸 출력과 정답 사이의 차이를 **손실 함수**로 수치화합니다. <br>예: 평균 제곱 오차(MSE), 교차 엔트로피 등 | **학생이 시험을 보고 정답과 얼마나 틀렸는지 점수를 매기는 것** |
| **5. 가중치 조정** | 손실을 줄이기 위해 **가중치**(모델 내부 파라미터)를 조금씩 바꿉니다. <br>→ **경사 하강법(gradient descent)** 이라는 방법을 주로 사용합니다.<br>  - **역전파(backpropagation)** : 손실이 각 가중치에 얼마나 영향을 미치는지(기울기)를 뒤로 거슬러 올라가며 계산. | **학생이 틀린 문제를 다시 풀면서 “왜 틀렸지?”를 생각하고, 다음엔 같은 실수를 안 하도록 공부 방법을 고치는 것** |
| **6. 반복(에폭)** | 위 과정을 **전체 데이터**를 여러 번(에폭) 반복합니다. 매번 가중치가 조금씩 개선돼 손실이 점점 낮아집니

### 4) PromptTemplate + LLM(stream()) + StrOutputParser

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. 컴포넌트 정의 (llm 은 공통 설정 셀의 llm 사용)
prompt = PromptTemplate.from_template("You are an expert in AI Expert. \
                                      Answer the question. <Question>: {input}에 대해 쉽게 설명해주세요.")

# chain 연결 (LCEL)
chain = prompt | llm | StrOutputParser()

# 스트리밍 출력을 위한 요청
answer = chain.stream({"input": "인공지능 모델의 학습 원리"})
# 스트리밍 출력
#print(answer)

for token in answer:
    # 스트림에서 받은 데이터의 내용을 출력합니다. 줄바꿈 없이 이어서 출력하고, 버퍼를 즉시 비웁니다.
    print(token, end="", flush=True)

### 인공지능 모델이 “학습”하는 원리  
(쉽게 이해할 수 있도록 비유와 함께 설명합니다)

---

## 1. 기본 아이디어: **데이터 → 모델 → 예측 → 피드백 → 개선**
| 단계 | 무슨 일? | 비유 |
|------|----------|------|
| **데이터 수집** | 모델에게 보여줄 예시(입력)와 정답(정답 레이블)을 모음 | 선생님이 학생에게 문제와 해답을 줌 |
| **모델 초기화** | 처음엔 무작위(아무것도 모르는) 상태 | 빈 종이에 아무것도 적히지 않은 상태 |
| **예측** | 현재 모델이 입력을 보고 “정답”을 추측 | 학생이 문제를 풀어봄 |
| **오차(손실) 계산** | 추측이 정답과 얼마나 다른지 수치화 | 답이 틀렸을 때 “점수 차이”를 확인 |
| **피드백(학습)** | 오차를 줄이도록 모델을 조금씩 조정 | 틀린 부분을 교정해가며 다시 공부 |
| **반복** | 위 과정을 여러 번 반복 → 점점 정확해짐 | 반복 학습을 통해 실력이 향상 |

---

## 2. 핵심 개념 3가지

### ① **손실 함수 (Loss Function)**
- **무엇을 하는가?** 모델이 만든 예측과 실제 정답 사이의 차이를 숫자로 나타냅니다.
- **예시**: 평균 제곱 오차(MSE), 교차 엔트로피(Cross‑Entropy) 등  
- **비유**: 시험 점수와 실제 정답 사이의 “점수 차이”를 계산하는 것.

### ② **경사 하강법 (Gradient Descent)**
- **무엇을 하는가?** 손실을 최소화하도록 모델의 파라미터(가중치)를 조금씩 바꿉니다.
- **핵심 아이디어**: 손실이 가장 크게 감소하는 방향(기울기)을 찾아서 그 방향으로 이동.
- **비유**: 언덕 위에 서서 가장 가파른 아래쪽으로 걸어 내려가면 최저점(골짜기)에 도착하는 것과 비슷.

### ③ **역전파 (Backpropagation)**
- **무엇을 하는가?** 손실이 어떻게 각 파라미터에 영향을 주는지(기울기)를 계산해 줍니

##### 2) Multiple Chains
* Multi Chain을 활용한 영화 추천 및 줄거리 요약 

In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
prompt1 = ChatPromptTemplate.from_template("{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.")

# Step 2: 추천된 영화의 줄거리를 요약
prompt2 = ChatPromptTemplate.from_template("{movie} 추천한 영화의 제목을 먼저 알려주시고,\
                                            줄을 바꾸어서 영화의 정보(제목,감독,캐스팅,줄거리)를 알려 주세요")

# 영화 예제용 모델 (공통 설정 셀의 get_llm 사용)
llm = get_llm(GROQ_MODEL)

# 체인 1: 영화 추천 (입력: 장르 → 출력: 영화 제목)
chain1 = prompt1 | llm | StrOutputParser()

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
movie = chain1.invoke({"genre": "Drama"})  # 영화 제목 얻기

print(type(movie))
print(" ===> 추천된 영화:")  # movie 값 출력
print(movie)


<class 'langchain_core.messages.base.TextAccessor'>
 ===> 추천된 영화:
**영화 추천: 《그린 북 (Green Book)》 (2018)**  

- **감독**: 피터 패럴리 (Peter Farrelly)  
- **주연**: 비고 모텐슨, 마하샬라 알리  
- **줄거리**: 1960년대 미국 남부를 무대로, 흑인 피아니스트 돈 셜리와 그의 백인 운전사 토니 리프가 투어를 떠나면서 겪는 문화·인종 갈등과 우정을 그린 실화 기반 드라마.  
- **추천 이유**  
  1. **감동적인 인간 관계** – 서로 다른 배경을 가진 두 인물이 점차 이해와 존중을 배우는 과정이 따뜻하게 다가옵니다.  
  2. **역사적 통찰** – 당시 인종 차별 현실을 사실적으로 보여주면서도 희망적인 메시지를 전합니다.  
  3. **수상 경력** – 아카데미 시상식에서 작품상·감독상·각본상·조연남우상 등 3관왕을 차지한 만큼 완성도가 높습니다.  
  4. **시청하기 쉬운 길이** – 약 2시간 10분 정도로 부담 없이 감상할 수 있습니다.  

드라마 장르를 좋아하신다면, 인간미와 사회적 메시지를 동시에 즐길 수 있는 《그린 북》을 한 번 보세요! 🎬


In [13]:
# 체인 2: 줄거리 요약 (입력: 영화 제목 → 출력: 줄거리)
chain2 = (
    {"movie": chain1}  # chain1의 출력을 movie 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)
print(chain2)

# 실행: "SF" 장르의 영화 추천 및 줄거리 요약
response = chain2.invoke({"genre": "Drama"})
print("\n🔹 영화 줄거리 요약:\n", response) 

first={
  movie: ChatPromptTemplate(input_variables=['genre'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['genre'], input_types={}, partial_variables={}, template='{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.'), additional_kwargs={})])
         | ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x000002BDB3B00AD0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002BDB43F3320>, root_client=<openai.OpenAI object at 0x000002BDB43F0FB0>, root_async_client=<openai.AsyncOpenAI object at 0x000002BDB43F0DD0>, model_name='openai/gpt-oss-120b', temperature=0.7, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.groq.com/openai/v1', stream_chunk_timeout=120.0)
         | StrOutputParser()
} middle=[ChatPromptTemplate

##### chain1과 chain2에서 영화 제목이 일관되게 전달 되도록 변경 

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from pprint import pprint

# Step 1: 사용자가 입력한 장르에 따라 영화 추천
prompt1 = ChatPromptTemplate.from_template("{genre} 장르에서 추천할 만한 영화를 한 편 알려주세요.")

# Step 2: 추천된 영화의 줄거리를 요약
prompt2 = ChatPromptTemplate.from_template("{movie} 추천한 영화의 제목을 먼저 알려주시고, 줄을 바꾸어서 영화의 정보(제목,감독,캐스팅,줄거리)를 알려 주세요.")

llm = get_llm(GROQ_MODEL)

# 체인 1: 영화 추천 (입력: 장르 → 출력: 영화 제목)
chain1 = prompt1 | llm | StrOutputParser()

# 체인 2: 줄거리 요약 (입력: 영화 제목 → 출력: 줄거리)
chain2 = (
    {"movie": chain1}  # chain1의 출력을 movie 변수로 전달
    | prompt2
    | llm
    | StrOutputParser()
)

# 실행: "Drama" 장르의 영화 추천 및 줄거리 요약
response = chain2.invoke({"genre": "Drama"})
print("\n🔹 영화 줄거리 요약:")
pprint(response)


🔹 영화 줄거리 요약:
('그린 마일 (The Green Mile, 1999)  \n'
 '\n'
 '**제목**: 그린 마일 (The Green Mile, 1999)  \n'
 '**감독**: 프랭크 다라본트  \n'
 '**캐스팅**: 톰 행크스, 마이클 클라크 딘, 데이빗 모스, 베라 파머, 제프리 디언 등  \n'
 '**줄거리**: 1930년대 미국 남부 교도소의 사형수 독방, 일명 ‘그린 마일’을 배경으로 간수 폴(톰 행크스)이 인간적인 대우를 '
 '시도한다. 그 과정에서 신비로운 치유 능력을 가진 죄수 존 코피(마이클 클라크 딘)와 특별한 인연을 맺게 되며, 인간성, 정의, 사랑, '
 '그리고 삶과 죽음의 의미를 깊이 있게 탐구한다.')


In [15]:
# 모델 설정 (공통 설정 셀의 get_llm 사용)
model = get_llm(GROQ_MODEL)

# 멀티 체인 실행 함수
def movie_chain(genre):
    print(f"\n🔹 1단계: '{genre}' 장르 영화 추천")
    
    # 체인 1: 영화 추천
    response1 = model.invoke(f"{genre} 장르에서 유명한 영화 제목 하나만 알려줘")
    movie = response1.content.strip()
    print(f"   → 추천 영화: {movie}")
    
    print(f"\n 2단계: '{movie}' 영화 정보 조회")
    
    # 체인 2: 영화 정보
    response2 = model.invoke(f"{movie} 영화에 대해 간단히 알려줘. 줄거리, 감독, 출연진 포함해서")
    info = response2.content
    print(f"   → 영화 정보:\n{info}")
    
    return movie, info

# 실행
genre = input("원하는 장르를 입력하세요: ")
movie, info = movie_chain(genre)
print(f"\n 최종 결과: '{movie}' 영화가 추천되었습니다!")


🔹 1단계: '액션' 장르 영화 추천
   → 추천 영화: 다음은 액션 장르에서 널리 알려진 영화 제목입니다:

**"다이 하드 (Die Hard)"**

 2단계: '다음은 액션 장르에서 널리 알려진 영화 제목입니다:

**"다이 하드 (Die Hard)"**' 영화 정보 조회
   → 영화 정보:
**다이 하드 (Die Hard, 1988)**  

| 항목 | 내용 |
|------|------|
| **감독** | 존 맥티어넌 (John McTiernan) |
| **주연 배우** | • 브루스 윌리스 (Bruce Willis) – 주인공 **존 맥클레인** <br>• Alan Rickman – 악당 **한스 그루버** <br>• Bonnie Bedelia – 맥클레인의 아내 **홀리** <br>• Reginald VelJohnson – 경찰관 **Al Powell** <br>• William Harris – 부동산 경매인 **Harry Ellis** |
| **줄거리 (간단히)** | 1995년 크리스마스 이브, 뉴욕 경찰관 **존 맥클레인**은 LA에서 아내 **홀리**와 함께 로스앤젤레스에 있는 고층 빌딩 ‘Nakatomi Plaza’에서 열리는 기업 파티에 참석한다. 빌딩이 독일 무장 테러리스트 **한스 그루버**와 그의 부하들에게 점령당하고, 인질을 잡히면서 상황은 급변한다. 경찰이 현장에 도착하기 전까지, 맥클레인은 혼자서 건물 안을 누비며 테러리스트들의 계획을 저지하고 인질들을 구하려고 사투를 벌인다. 총격전, 폭발, 그리고 기발한 함정이 뒤섞인 액션이 이어지며, 결국 맥클레인은 테러리스트들을 물리치고 인질들을 구해낸다. |
| **특징** | * 1980년대 액션 영화의 아이콘이 된 작품으로, “한 남자가 혼자서 악당을 무너뜨린다”는 전형적인 구조를 확립함.<br>* 브루스 윌리스는 이 영화로 전설적인 액션 히어로 ‘존 맥클레인’이라는 캐릭터를 탄생시켰다.<br>* 알란 리크먼의 카리스마 넘치는 악역 연기가 큰 호평을 받았다.<br>* 